# Projet 8 - Construire et testez une infrastructure de données

## Mission

Vous êtes nouvellement embauché comme Data Engineer dans l’entreprise GreenAndCoop, un fournisseur coopératif français d'électricité d'origine renouvelable dans les Hauts-de-France. 

En tant que Data Engineer, votre travail est de fournir quotidiennement des données météorologiques de qualité aux Data Scientists pour leurs modèles de prévision. La prévision de demande est un enjeu stratégique  pour GreenCoop. Elle permet de  :

1. équilibrer le réseau. GreenCoop doit s’assurer que la production et la consommation d’électricité sont équilibrées en temps réel pour éviter des pénalités financières.

2. optimiser la production. Les sources d’énergie renouvelable, comme le solaire et l’éolien, sont intermittentes. En prévoyant la demande, GreenCoop peut planifier plus efficacement l’utilisation de ses ressources et maximiser l’efficacité de la production.

3. maîtriser ses coûts. Une prévision précise de la demande permet de réduire les coûts liés à l’achat d’électricité sur le marché de gros, en évitant les achats d’urgence à des prix élevés.

Pour gagner en précision dans leurs prévisions, les Data Scientists de GreenCoop ont lancé un nouveau projet qui s’appelle Forecast 2.0. L’objectif de celui-ci est d’intégrer de nouvelles sources de données dans leurs algorithmes comme celles issues de stations météorologiques semi-professionnelles. 

Vous êtes intégré à ce projet en tant que Data Engineer. Votre mission consiste à concevoir, implémenter et industrialiser un pipeline de données de type ELT permettant d’ingérer, transformer et fiabiliser ces nouvelles données météorologiques. Les données proviennent de différents réseaux (open data, stations InfoClimat, stations Weather Underground, etc.) et sont transmises à des fréquences variables (toutes les 10, 15 ou 30 minutes).

En consultant  vos mails ce matin, vous découvrez un message d’Ouly, la cheffe du projet Forecast 2.0 et membre de l’équipe Data Science :

De : Ouly

À : Moi

Objet : Forecast 2.0 - Intégration de nouvelles sources de données dans la BDD

Salut,

J'espère que tu vas bien !

Je suis ravie que tu nous rejoignes sur ce beau projet. Tu vas nous aider à améliorer nos algorithmes de prévision en les enrichissant de nouvelles sources de données. On avait besoin d’un bon Data Engineer pour mettre en place un pipeline fiable, automatisé et maintenable.

Suite à notre discussion, tu trouveras en pièces jointes les liens vers plusieurs sources de données que nous avons identifiées. Ces nouvelles sources vont nous aider. En effet, elles se trouvent dans des localisations où notre modèle de prévision de la demande d'électricité est moins performant faute de relevés précis par les stations météorologiques officielles. Ces sources sont le réseau InfoClimat et le réseau Weather Underground. 

Les formats des données sont hétérogènes. Nous souhaitons conserver toutes les informations des sources que je t’ai envoyées dans la base de données finale. Pour faciliter l’ingestion et la synchronisation de ces différentes sources, nous te recommandons de t’appuyer sur un outil d’intégration tel qu’Airbyte, afin de centraliser les flux vers PostgreSQL avant transformation.

Nous souhaitons conserver toutes les informations des sources que je t’ai envoyées dans la base de données finale. Il faut donc les structurer dans un schéma de données cohérent adapté à des usages analytiques et à nos besoins de modélisation.

Pour la transformation, la documentation et les contrôles de qualité des données, nous souhaitons nous appuyer sur DBT, afin de centraliser la logique de transformation directement dans la base de données, de versionner les modèles et de garantir la fiabilité des données mises à disposition.

Je ne sais pas si tu es au courant, mais notre DSI nous impose de travailler avec AWS. Nous souhaitons que ces données soient stockées dans une base de données PostgreSQL hébergée sur AWS afin qu’elles puissent être facilement exploitées par les équipes Data Science, notamment dans nos environnements de machine learning (pour info, on utilise SageMaker pour nos travaux de ML donc on pourra se connecter directement).

Nous avons une réunion de l’équipe projet dans 1 mois. Peux-tu préparer une présentation pour nous expliquer ta démarche ainsi qu’une démonstration de la solution mise en place ? 

Les éléments qui nous intéressent sont les suivants :

* Le schéma de la base de données, et la manière dont les données peuvent être interrogées,

* le processus de collecte, de transformation et de test des données, notamment via DBT (on utilise souvent des logigrammes pour présenter nos processus),

* L’architecture globale de la base de données,

* La stack technique utilisée (les outils/services web utilisés),

* Les mécanismes de contrôle de la qualité des données (taux d’erreurs),

* Le délai de mise à disposition des données.

Bon courage à toi et à très bientôt,

Ouly

P.J: Liens vers les sources de données identifiées

1. Stations météorologiques du réseau InfoClimat (Bergues, Hazebrouck, Armentières, Lille-Lesquin)

2. Station amateur Weather Underground à Ichtegem, Belgique

3. Station amateur Weather Underground à La Madeleine, France



## Préparez votre environnement de travail

Docker compose pour le déployement de PostgreSQL. Voici le contenu du compose.yaml:

```
services:

  postgres:
    image: postgres:18
    container_name: postgres
    restart: unless-stopped
    environment:
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: Seb+postgresql1

    ports:
      - "5433:5432" # en local, le port 5432 est déjà utilisé par PostgreSQL
    volumes:
      - "./postgres_data:/var/lib/postgresql:rw"

  pgadmin:
     image: dpage/pgadmin4
     environment:
       - PGADMIN_DEFAULT_EMAIL=admin@admin.com
       - PGADMIN_DEFAULT_PASSWORD=root
     ports:
       - "8080:80"

# Airbyte déployé depuis Docker Compose est déprécié depuis 2024 au profit d’abctl (basé sur Docker). abctl est installé en local.
  # airbyte:
    # image: airbyte/airbyte:latest
    # container_name: airbyte
    # restart: unless-stopped
    # ports: 
      # - "8000:8000"
    # depends_on:
       #- postgres
```





`docker compose up`

Vérification de la base PostgreSQL.

```
(base) C:\Users\sebas>docker exec -it postgres psql -U postgres
psql (18.4 (Debian 18.4-1.pgdg13+1))
Type "help" for help.

postgres=# SELECT current_database();
 current_database
------------------
 postgres
(1 row)

``` 
Lancement de pgAdmin via http://localhost:8080/ dans la barre de l'explorateur internet


Récupération de la version du abclt sur le site `https://docs.airbyte.com/platform/using-airbyte/getting-started/oss-quickstart`

```
(base) C:\Users\sebas>abctl version

(base) C:\Users\sebas>abctl local install
```
Récupérer le mot de passe pour accéder à Airbyte.
```
(base) C:\Users\sebas>abctl local credentials
  INFO    Using Kubernetes provider:
            Provider: kind
            Kubeconfig: C:\Users\sebas\.airbyte\abctl\abctl.kubeconfig
            Context: kind-airbyte-abctl
 SUCCESS  Retrieving your credentials from 'airbyte-auth-secrets'
  INFO    Credentials:
            Email: [not set]
            Password: PQ3Lc9U0E5g4gdNZuiyOwni437dV4oJv
            Client-Id: 01670c34-d77a-4b64-94c4-9004ff627116
            Client-Secret: AJOV3rBUkRby8fK9wj69LR4a8RrtL6If
```
Rajouter une adresse mail (il n'est pas nécessaire qu'elle soit réelle pour un exercice).
```
(base) C:\Users\sebas>abctl local credentials --email sebastien@exemple.com
  INFO    Using Kubernetes provider:
            Provider: kind
            Kubeconfig: C:\Users\sebas\.airbyte\abctl\abctl.kubeconfig
            Context: kind-airbyte-abctl
  INFO    Updating email for authentication
 SUCCESS  Email updated
 SUCCESS  Retrieving your credentials from 'airbyte-auth-secrets'
  INFO    Credentials:
            Email: sebastien@exemple.com
            Password: PQ3Lc9U0E5g4gdNZuiyOwni437dV4oJv
            Client-Id: 01670c34-d77a-4b64-94c4-9004ff627116
            Client-Secret: AJOV3rBUkRby8fK9wj69LR4a8RrtL6If
```

lancer airbyte en tapant dans la barre de l'explorateur: `http://localhost:8000/`

si cela ne fonctionne pas, faire :`abctl local install --insecure-cookies` et relancer.

Se connecter avec l'email



## Récupérer les données météorologiques avec Airbyte

### Créer la database dans PostgreSQL en passant par pgAdmin.

![creation_db](create_database_pgadmin.png)


### Configurer la destination PostgreSQL dans Airbyte

Dans Airbyte, cliquer sur `Destinations` puis `+ New destination` puis `PostgreSQL`

Compléter les champs suivants:

* Destination name: Postgres
* Host : host.docker.internal
* Port : 5433
* Database : weather_data
* Schema : raw
* Username/Password : les identifiants de connexion à PostgreSQL définit dans le docker compose
* SSL Mode : disable

![configuration_destination](config_destination_postgres.png)


La destination a été créée avec succès:


![destination_creee](destination_creee.png)

### Configurer la source dans Airbyte

Avec abctl on ne peut pas simplement pointer vers un dossier local Windows pour la source des fichiers. 

On peut, via GitHub, mettre à disposition les fichiers Excel et JSON et laisser Airbyte les récupérer avec une URL raw.

Exemple:

`https://github.com/Sebules/Project_8_OpenClassrooms_Data_Engineer/raw/refs/heads/main/donnees/Weather_Underground_Ichtegem_BE.xlsx`

Création d'une source:

![config_source_excel.png](config_source_excel.png)


La source a été créée:


![source_excel_creee.png](source_excel_creee.png)

### Créer la connexion

cliquer sur `Create a connection`

![click_add_connection.png](click_add_connection.png)

Ensuite, cliquer sur `Postgres`

![click_postgres.png](click_postgres.png)


Choisir le sync mode full refresh orverwrite pour garder les données raw

![resultat_chose_sync_mode_full_refresh_overwrite.png](resultat_chose_sync_mode_full_refresh_overwrite.png)


Choisir schedule type manual

![resultat_chose_schedule_manual.png](resultat_chose_schedule_manual.png)


Cliquer sur `set up connection`. Le résultat en image.

![resultat_set_up_connection.png](resultat_set_up_connection.png)


Il faut faire la synchronisation:

![click_sync_now.png](click_sync_now.png)

Résultat de synchro qui s'est bien passée:

![click_sync_now_resultat.png](click_sync_now_resultat.png)


### Vérification dans PostgreSQL via pgAdmin

```
SET search_path TO raw;

SELECT * FROM "Weather_Underground_Ichtegem_BE" LIMIT 10;
```

![verification_table_belgique.png](verification_table_belgique.png)
